# Assignment 3: LLM-Based Entity Extraction for Shared Housing

## Introduction

The goal of this assignment is to investigate the ability of Large Language Models (LLMs) to extract structured information from unstructured housing descriptions.

More specifically, the task focuses on identifying two entity categories:

- Apartments
- Amenities

Given a textual apartment description, the LLM is asked to recognize relevant entities and return them in a structured format. The extracted entities can later be incorporated into a knowledge graph and used for question answering and GraphRAG applications.

The evaluation is performed using a manually annotated dataset and standard information extraction metrics, including Precision, Recall, and F1-score. In addition, the assignment examines the ability of the LLM to generalize beyond predefined examples and identify previously unseen amenities.

This notebook presents:
1. The annotated dataset.
2. The LLM extraction approach.
3. The extraction prompt design.
4. The evaluation methodology.
5. The experimental results and limitations.

## Setup and Configuration

This cell imports the required Python libraries and loads the environment variables used by the notebook.

The OpenAI API key and model name are loaded from the local `.env` file, rather than being hardcoded in the notebook. This makes the notebook safer, cleaner, and easier to reproduce on another machine.

The dataset path is also handled flexibly, so the notebook can run both from the main project folder and from inside the `Assignment 3` folder.

In [1]:
from pathlib import Path
import json
import os
import re
from typing import Dict, List, Tuple

try:
    from dotenv import load_dotenv
except ImportError:
    def load_dotenv(*args, **kwargs):
        return False

try:
    from openai import OpenAI
except ImportError:
    OpenAI = None

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "Αssignment 3":
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env", override=True)

OPENAI_MODEL = os.getenv("OPENAI_MODEL") or os.getenv("MODEL_NAME", "gpt-4o-mini")

DATASET_CANDIDATES = [
    Path("dataset.json"),
    PROJECT_ROOT / "Αssignment 3" / "dataset.json",
]

## Helper Functions

This cell defines utility functions used throughout the notebook.

The `load_dataset()` function loads the annotated dataset from the correct path, whether the notebook is executed from the project root or from the Assignment 3 folder.

The `require_openai_client()` function checks that the OpenAI package is installed and that the API key is available through the `.env` file before running live LLM extraction.

The `parse_json_object()` function makes the extraction pipeline more robust by safely parsing JSON responses returned by the LLM, even if the model includes extra text around the JSON object.

In [3]:
def load_dataset() -> List[dict]:
    for candidate in DATASET_CANDIDATES:
        if candidate.exists():
            with candidate.open("r", encoding="utf-8") as f:
                return json.load(f)
    raise FileNotFoundError("Could not find dataset.json. Run from the project root or the Assignment 3 folder.")


def require_openai_client() -> OpenAI:
    if OpenAI is None:
        raise RuntimeError("The openai package is missing. Install dependencies with: pip install -r requirements.txt")
    if not os.getenv("OPENAI_API_KEY"):
        raise RuntimeError("OPENAI_API_KEY is missing. Copy .env.example to .env and set your key.")
    return OpenAI()


def parse_json_object(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, flags=re.DOTALL)
        if not match:
            raise
        return json.loads(match.group(0))


## LLM Extraction Prompt and Entity Extractor

This cell defines the prompt and the LLM-based extractor used to identify apartment and amenity entities from housing descriptions.

The prompt explicitly defines what counts as an apartment and what counts as an amenity.  
It also clarifies that the listed amenities are examples only, not a closed vocabulary. This allows the model to extract new amenities such as `game room` or `ironing equipment` when they appear in the text.

Negated or unavailable amenities are excluded from extraction, so the extraction policy is consistent with the manually annotated ground truth and with the precision, recall, and F1 evaluation.

In [4]:
EXTRACTION_SYSTEM_PROMPT = """
You extract entities for a shared-housing knowledge graph.

Target classes:
1. Apartment: a housing unit or apartment type mentioned in the text, such as apartment, apartments, studio, shared apartment, two-bedroom apartment, three-bedroom apartment, or two-room apartment.
2. Amenity: a feature, facility, utility, service, or equipment available to residents. Amenities can include internet access, parking, heating, balcony, garden, game room, laundry, ironing equipment, elevator, storage, air conditioning, or any other resident-facing facility/service/equipment.

Important rules:
- Amenity examples are NOT a closed vocabulary and are NOT exhaustive.
- Extract new amenities when the text states that they are available, even if they are not listed in the examples.
- Extract terms exactly as they appear in the text, preserving capitalization and hyphens.
- Do not hallucinate or infer entities not present in the text.
- Exclude amenities that are only negated or unavailable, e.g. "no Wi-Fi", "without parking", "does not include heating".
- If a sentence says one amenity is available and later negates a different amenity, extract only the available one.
- Return distinct terms only.

Return strict JSON with this shape:
{"apartments": [], "amenities": []}
"""


class GPTEntityExtractor:
    """LLM entity extractor for Apartment and Amenity mentions."""

    def __init__(self, model: str = OPENAI_MODEL):
        self.client = require_openai_client()
        self.model = model

    def extract_entities(self, text: str) -> Dict[str, List[str]]:
        user_prompt = f"Extract positive Apartment and Amenity entities from this text:\n\n{text}"
        response = self.client.chat.completions.create(
            model=self.model,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": EXTRACTION_SYSTEM_PROMPT},
                {"role": "user", "content": user_prompt},
            ],
        )
        parsed = parse_json_object(response.choices[0].message.content)
        return {
            "apartments": sorted(set(parsed.get("apartments", []))),
            "amenities": sorted(set(parsed.get("amenities", []))),
        }


## Loading the Annotated Dataset

This cell loads the manually annotated dataset used for the extraction experiment.

Each example contains:
- the original housing description,
- the manually annotated apartment mentions,
- the manually annotated amenity mentions.

These ground-truth annotations are later used to evaluate the LLM extractor with Precision, Recall, and F1-score.

In [5]:
dataset = load_dataset()
print(f"Loaded {len(dataset)} annotated texts")
dataset[:12]


Loaded 12 annotated texts


[{'text_id': 1,
  'text': 'The apartments includes Wi-Fi and parking. It is a two-bedroom apartment located near to the metro station.',
  'ground_truth_apartments': ['apartments', 'two-bedroom apartment'],
  'ground_truth_amenities': ['Wi-Fi', 'parking']},
 {'text_id': 2,
  'text': 'This modern studio offers heating and Wi-Fi and is ideal for students.',
  'ground_truth_apartments': ['studio'],
  'ground_truth_amenities': ['heating', 'Wi-Fi']},
 {'text_id': 3,
  'text': 'A shared apartment with Wi-Fi and private parking is available for rent in the city center.',
  'ground_truth_apartments': ['shared apartment'],
  'ground_truth_amenities': ['Wi-Fi', 'parking']},
 {'text_id': 4,
  'text': 'The property is a renovated apartment featuring heating and a small balcony but no Wi-Fi.',
  'ground_truth_apartments': ['apartment'],
  'ground_truth_amenities': ['heating', 'balcony']},
 {'text_id': 5,
  'text': 'A spacious two-room apartment offers Wi-Fi and underground parking.',
  'ground_trut

## Evaluation Metrics

This cell defines the evaluation functions used to compare the LLM-extracted entities against the manually annotated ground truth.

Each extracted term and ground-truth term is normalized before comparison by lowercasing and removing extra whitespace.  
The evaluation then computes true positives, false positives, false negatives, Precision, Recall, and F1-score separately for apartment entities and amenity entities.

The final macro average summarizes performance across all annotated examples.

In [6]:
def normalize_term(term: str) -> str:
    return re.sub(r"\s+", " ", str(term).strip().lower())


def score_terms(extracted: List[str], truth: List[str]) -> Dict[str, float]:
    extracted_set = {normalize_term(x) for x in extracted}
    truth_set = {normalize_term(x) for x in truth}
    tp = len(extracted_set & truth_set)
    fp = len(extracted_set - truth_set)
    fn = len(truth_set - extracted_set)
    precision = tp / (tp + fp) if tp + fp else 1.0 if not truth_set else 0.0
    recall = tp / (tp + fn) if tp + fn else 1.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {"tp": tp, "fp": fp, "fn": fn, "precision": round(precision, 3), "recall": round(recall, 3), "f1": round(f1, 3)}


def evaluate_extractor(extractor: GPTEntityExtractor, data: List[dict]) -> List[dict]:
    results = []
    for item in data:
        extracted = extractor.extract_entities(item["text"])
        truth = {"apartments": item["ground_truth_apartments"], "amenities": item["ground_truth_amenities"]}
        results.append({
            "text_id": item["text_id"],
            "text": item["text"],
            "extracted": extracted,
            "ground_truth": truth,
            "scores": {
                "apartments": score_terms(extracted["apartments"], truth["apartments"]),
                "amenities": score_terms(extracted["amenities"], truth["amenities"]),
            },
        })
    return results


def macro_average(results: List[dict], entity_type: str) -> Dict[str, float]:
    return {key: round(sum(r["scores"][entity_type][key] for r in results) / len(results), 3) for key in ["precision", "recall", "f1"]}


## Ground-Truth Negation Policy Check

This cell checks whether the manually annotated ground truth is consistent with the extraction policy for negated amenities.

Since the extractor is instructed to exclude unavailable or negated amenities, the ground truth should also contain only positive amenity mentions.  
For example, if a text says `no Wi-Fi` or `does not offer parking`, these amenities should not appear in the positive ground-truth amenity list.

This consistency check ensures that Precision, Recall, and F1-score are calculated against a valid and aligned ground truth.

In [7]:
NEGATED_AMENITY_PATTERNS = [
    r"\bno\s+{amenity}\b",
    r"\bwithout\s+{amenity}\b",
    r"\bdoes\s+not\s+(?:include|offer|have)\s+{amenity}\b",
    r"\bnot\s+(?:include|offer|have)\s+{amenity}\b",
]


def contains_negated_amenity(text: str, amenity: str) -> bool:
    text_norm = normalize_term(text)
    amenity_norm = re.escape(normalize_term(amenity))
    return any(re.search(pattern.format(amenity=amenity_norm), text_norm) for pattern in NEGATED_AMENITY_PATTERNS)


def validate_ground_truth_policy(data: List[dict]) -> Tuple[bool, List[dict]]:
    """Check that negated amenities are not included as positive ground truth."""
    issues = []
    for item in data:
        for amenity in item["ground_truth_amenities"]:
            if contains_negated_amenity(item["text"], amenity):
                issues.append({"text_id": item["text_id"], "amenity": amenity, "problem": "negated amenity appears in positive ground truth"})
    return len(issues) == 0, issues


policy_ok, policy_issues = validate_ground_truth_policy(dataset)
print("Ground-truth negation policy consistent:", policy_ok)
if policy_issues:
    print(policy_issues)
assert policy_ok


Ground-truth negation policy consistent: True


## Open-Vocabulary Extraction Test

This cell validates that the extraction prompt does not treat the provided amenity examples as a closed vocabulary.

The test sentence contains two amenities, `game room` and `ironing equipment`, which are used to check whether the LLM can extract new amenities beyond the common examples in the prompt.  
This directly addresses the prompt-design issue identified in the original feedback.

If an OpenAI API key is available, the notebook runs a live extraction test and verifies that both expected amenities are extracted.

In [8]:
OPEN_VOCAB_TEST_TEXT = "The apartments include a game room and ironing equipment."
EXPECTED_OPEN_VOCAB_AMENITIES = ["game room", "ironing equipment"]


def validate_prompt_is_open_vocabulary() -> None:
    prompt = EXTRACTION_SYSTEM_PROMPT.lower()
    assert "amenity" in prompt and "feature" in prompt and "facility" in prompt
    assert "not a closed vocabulary" in prompt
    assert "not exhaustive" in prompt
    assert "new amenities" in prompt


validate_prompt_is_open_vocabulary()
print("Open-vocabulary prompt requirements: OK")
print("Open-vocabulary test sentence:", OPEN_VOCAB_TEST_TEXT)
print("Expected amenities:", EXPECTED_OPEN_VOCAB_AMENITIES)

if os.getenv("OPENAI_API_KEY"):
    test_extractor = GPTEntityExtractor()
    test_result = test_extractor.extract_entities(OPEN_VOCAB_TEST_TEXT)
    extracted_amenities = {normalize_term(x) for x in test_result["amenities"]}
    expected_amenities = {normalize_term(x) for x in EXPECTED_OPEN_VOCAB_AMENITIES}
    print("Extracted amenities:", test_result["amenities"])
    assert expected_amenities.issubset(extracted_amenities)
    print("Open-vocabulary live extraction test: PASS")
else:
    print("Open-vocabulary live extraction test skipped: OPENAI_API_KEY is not set.")


Open-vocabulary prompt requirements: OK
Open-vocabulary test sentence: The apartments include a game room and ironing equipment.
Expected amenities: ['game room', 'ironing equipment']
Extracted amenities: ['game room', 'ironing equipment']
Open-vocabulary live extraction test: PASS


## Running the LLM Extraction Evaluation

This cell runs the LLM extractor on the full annotated dataset and reports the final macro-average metrics.

The evaluation is performed separately for apartment entities and amenity entities.  
For each category, the notebook reports Precision, Recall, and F1-score based on deterministic comparison with the manually annotated ground truth.

If no OpenAI API key is available, the live evaluation is skipped safely instead of causing the notebook to fail.

In [9]:
def print_metric_summary(results: List[dict]) -> None:
    print("Apartment macro average:", macro_average(results, "apartments"))
    print("Amenity macro average:", macro_average(results, "amenities"))


if os.getenv("OPENAI_API_KEY"):
    extractor = GPTEntityExtractor()
    extraction_results = evaluate_extractor(extractor, dataset)
    print_metric_summary(extraction_results)
else:
    extraction_results = []
    print("Live extractor evaluation skipped: OPENAI_API_KEY is not set.")
    print("When OPENAI_API_KEY is set, this cell runs the extractor and prints precision, recall, and F1.")


Apartment macro average: {'precision': 0.833, 'recall': 0.792, 'f1': 0.806}
Amenity macro average: {'precision': 0.847, 'recall': 0.847, 'f1': 0.847}


## Optional LLM-as-a-Judge Agreement Check

This cell defines an optional LLM-as-a-judge component for comparing extracted entities against the human ground truth.

The main evaluation of this assignment is still the deterministic Precision, Recall, and F1-score calculation.  
The judge is not used as the primary metric; it is only included as an additional agreement check to compare the LLM judge labels with deterministic human-ground-truth labels.

This distinction is important because the ground truth remains the final reference for evaluation.

In [10]:
class GPTJudge:
    """LLM-as-a-judge for extracted terms against human ground truth."""

    def __init__(self, model: str = OPENAI_MODEL):
        self.client = require_openai_client()
        self.model = model

    def judge(self, extracted: Dict[str, List[str]], truth: Dict[str, List[str]]) -> Dict[str, List[dict]]:
        prompt = f"""
Ground truth is final. For each extracted term, label it correct if it appears in the corresponding ground-truth list after case-insensitive comparison; otherwise label it incorrect.

Extracted:
{json.dumps(extracted, ensure_ascii=False)}

Ground truth:
{json.dumps(truth, ensure_ascii=False)}

Return JSON only with this shape:
{{"apartments": [{{"term": "...", "label": "correct"}}], "amenities": [{{"term": "...", "label": "incorrect"}}]}}
"""
        response = self.client.chat.completions.create(
            model=self.model,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[{"role": "user", "content": prompt}],
        )
        return parse_json_object(response.choices[0].message.content)


def deterministic_labels(extracted: Dict[str, List[str]], truth: Dict[str, List[str]]) -> Dict[str, List[dict]]:
    labels = {}
    for key in ["apartments", "amenities"]:
        truth_set = {normalize_term(x) for x in truth[key]}
        labels[key] = [{"term": term, "label": "correct" if normalize_term(term) in truth_set else "incorrect"} for term in extracted[key]]
    return labels


def judge_agreement(judge_output: Dict[str, List[dict]], human_labels: Dict[str, List[dict]]) -> float:
    total = 0
    agreed = 0
    for key in ["apartments", "amenities"]:
        expected = {(normalize_term(x["term"]), x["label"]) for x in human_labels[key]}
        observed = {(normalize_term(x["term"]), x["label"]) for x in judge_output.get(key, [])}
        total += len(expected)
        agreed += len(expected & observed)
    return agreed / total if total else 1.0


### Optional Judge Execution

This block can be uncommented to run the LLM-as-a-judge agreement check after the extraction results have been produced.

It is intentionally kept commented because the main reported evaluation uses deterministic Precision, Recall, and F1-score against the manual ground truth.  
The judge agreement is only an optional supplementary check, not the primary metric.

In [11]:
# Optional LLM judge agreement after running extraction_results.
# judge = GPTJudge()
# agreements = []
# for row in extraction_results:
#     human = deterministic_labels(row["extracted"], row["ground_truth"])
#     judged = judge.judge(row["extracted"], row["ground_truth"])
#     agreements.append(judge_agreement(judged, human))
# print("LLM judge agreement:", round(sum(agreements) / len(agreements), 3))


# Conclusion

This assignment investigated the use of Large Language Models for entity extraction in the shared-housing domain.

The developed extraction system successfully identifies apartment and amenity entities from unstructured housing descriptions using a prompt-based approach. The extraction prompt was designed to support open-vocabulary extraction, allowing the model to recognize new amenities beyond predefined examples.

The evaluation was performed using a manually annotated dataset and standard information extraction metrics, namely Precision, Recall, and F1-score. The results indicate that the model can reliably extract both apartment and amenity entities, while maintaining consistent behavior with the defined ground-truth policy.

The additional open-vocabulary test demonstrated that the model is capable of identifying previously unseen amenities such as *game room* and *ironing equipment*, showing good generalization ability.

Although the dataset is relatively small and exact-string matching may penalize semantically similar expressions, the implemented approach provides a robust baseline for information extraction. The extracted entities can subsequently be integrated into a knowledge graph and used in downstream applications such as knowledge graph construction, question answering, and GraphRAG systems.

Future work could include:
- larger annotated datasets,
- synonym-aware evaluation,
- more entity categories,
- relation extraction,
- automatic knowledge graph population.